In [41]:
import os
import torch
import torch.optim as optim
import torch.nn.functional as F
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from numpy.ma.core import outer
import torch.nn as nn
from torchvision import datasets, transforms



In [42]:
img = Image.open("1024_1024_test_image.png")
transform = transforms.ToTensor()
img_tensor = transform(img)
img_tensor = img_tensor.unsqueeze_(0)

In [43]:
img_tensor.size()


torch.Size([1, 3, 1024, 1024])

In [44]:
class OverlapPatchEmbedding_T(nn.Module):
    def __init__(self, in_channels, out_channels,out_image_size, kernel_size, stride, padding):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        self.norm = nn.LayerNorm((out_channels,out_image_size, out_image_size),eps=1e-05,elementwise_affine=True)

    def forward(self, x):
        x = self.proj(x)  # Convolution # Flatten and transpose for transformer
        x = self.norm(x)
        return x

In [55]:
import torch
import torch.nn as nn
from typing import Tuple

class OverlapPatchEmbedding_V(nn.Module):
    """
    Overlap patch embedding (SegFormer-style).
    - proj: Conv2d(in_channels -> embed_dim) with kernel/stride/padding (overlap if stride < kernel)
    - norm: LayerNorm applied on the channel dimension after flattening: (B, N, C)
    Returns:
      tokens: (B, N, embed_dim)
      H: spatial height after conv
      W: spatial width after conv
      feat_map: (B, embed_dim, H, W)  # optional, handy for decoder fusion
    """
    def __init__(self,in_channels: int, embed_dim: int = 32, kernel_size: int = 7, stride: int = 4, padding: int = 3):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=kernel_size, stride=stride, padding=padding,  bias=True)
        # LayerNorm normalized_shape should match the last dim of tokens: embed_dim
        self.norm = nn.LayerNorm(embed_dim, eps=1e-5)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, int, int, torch.Tensor]:
        """
        x: (B, in_channels, H, W)
        returns:
          tokens: (B, H'*W', embed_dim)
          H', W': spatial dims after conv
          feat_map: (B, embed_dim, H', W')
        """
        x = self.proj(x)                   # (B, embed_dim, H', W')
        B, C, H, W = x.shape

        # tokens for transformer-style blocks: (B, N, C)
        tokens = x.flatten(2).transpose(1, 2)  # (B, H'*W', C)
        tokens = self.norm(tokens)             # LN over channel dim

        # also return feature map (useful for decoder)
        feat_map = tokens.transpose(1, 2).reshape(B, C, H, W)

        return tokens, H, W, feat_map


In [58]:
from stem import OverlapPatchEmbedding # stem.py prod qual class with optional parameters
stem = OverlapPatchEmbedding(in_channels=3,embed_dim=64,kernel_size=7,stride=4,padding=3,norm="bn",gn_num_groups=1,activation="gelu",dropout=0.1)
s = stem(img_tensor)
p = {"token" :s[0].shape, "feat_map": s[3].shape}
p


{'token': torch.Size([1, 65536, 64]),
 'feat_map': torch.Size([1, 64, 256, 256])}

torch.Size([1, 32, 256, 256])

In [59]:
encoder = lambda x:object
preprocessor = lambda x:object

decoder = lambda x:object
tracker = lambda x:object
postprocessor = lambda x:object
Output = lambda x:object

class evit:
    def __init__(self,in1,in2):
        self.pre = preprocessor(2)
        self.stem = OverlapPatchEmbedding(3,3,512,4,3,0)
        self.encoder = encoder(4)
        self.decoder = decoder(5)
        self.post = postprocessor(5)
        self.tracker = tracker(5)
        self.out = Output(5)
    def forward(self,x,in2):
        y = x

        x = self.pre(x)
        x = self.stem(x)
        x = self.encoder(x)
        x = self.decoder(x)
        y = self.tracker(y,in2)



